# Object-Oriented Programming in JavaScript

Object-Oriented Programming (OOP) is a paradigm that organizes code into reusable objects containing data (**properties**) and behavior (**methods**).

While traditional languages like Java use class-based structures, JavaScript is inherently a **prototype-based** language. In modern JavaScript (ES6+), the `class` keyword provides familiar syntax that acts as "syntactic sugar" over the underlying prototypal inheritance system.

> See also: [[javascript-prototypes]]

---

## 🧱 Core Structural Blocks: Classes and Objects

Define blueprints with the `class` keyword and instantiate them with `new`.

```javascript
class Car {
  // The constructor initializes the object's properties
  constructor(brand, speed) {
    this.brand = brand; // "this" points to the created instance
    this.speed = speed;
  }

  // A method defining the behavior of the object
  accelerate() {
    console.log(`${this.brand} is driving at ${this.speed} km/h.`);
  }
}

const myCar = new Car("Tesla", 120);
myCar.accelerate(); // Tesla is driving at 120 km/h.
```

### Class fields (ES2022)

You can declare properties directly in the class body without touching the constructor:

```javascript
class Counter {
  count = 0;              // public instance field
  #secret = "hidden";     // private instance field
  static version = "1.0"; // static field, lives on the class itself

  static {                // static initialization block, runs once
    console.log(`Counter v${Counter.version} loaded`);
  }
}
```

⚠️ **Important:** class fields are created *per instance*, methods are shared on the prototype. A function assigned as a field (`handle = () => {...}`) is duplicated for every instance — that costs memory, but it's the standard trick for auto-binding `this` (see the `this` section below).

### Static members

Static methods and properties belong to the class, not to instances. Use them for factories, constants, and utilities that don't need instance state.

```javascript
class Temperature {
  static FREEZING_C = 0;

  constructor(celsius) {
    this.celsius = celsius;
  }

  // Factory / alternative constructor
  static fromFahrenheit(f) {
    return new Temperature((f - 32) * 5 / 9);
  }
}

const t = Temperature.fromFahrenheit(212);
t.celsius;                    // 100
t.fromFahrenheit;             // undefined — not on instances
Temperature.FREEZING_C;       // 0
```

---

## 🏛️ The Four Pillars of OOP in JavaScript

### 1. Encapsulation

Encapsulation bundles data and methods into a single unit while restricting direct outside access. JavaScript achieves this natively with private fields prefixed by `#`.

```javascript
class BankAccount {
  #balance; // Private. Not accessible outside the class.

  constructor(owner, initialBalance) {
    this.owner = owner;
    this.#balance = initialBalance;
  }

  // Public method providing controlled access
  deposit(amount) {
    if (amount > 0) this.#balance += amount;
  }

  // Getter — read like a property, computed like a method
  get balance() {
    return `Account balance: $${this.#balance}`;
  }
}

const account = new BankAccount("Alice", 1000);
account.deposit(500);
console.log(account.balance);  // Account balance: $1500
// account.#balance;           // SyntaxError: Private field must be declared in an enclosing class
```

#### Getters and setters

A setter lets you intercept assignment and validate it, keeping the clean property syntax:

```javascript
class Product {
  #price = 0;

  get price() {
    return this.#price;
  }

  set price(value) {
    if (typeof value !== "number" || value < 0) {
      throw new RangeError("Price must be a non-negative number");
    }
    this.#price = value;
  }
}

const p = new Product();
p.price = 49.99;   // calls the setter
p.price;           // 49.99 — calls the getter
// p.price = -5;   // RangeError
```

Note the naming: the private field is `#price` and the accessor is `price`. They can't share a name, which is exactly why the `#` prefix convention is convenient.

#### Private methods and brand checks

```javascript
class Session {
  #token;

  #isExpired() {          // private method
    return Date.now() > this.#token.exp;
  }

  static isSession(obj) {  // "brand check" — works even on subclasses
    return #token in obj;
  }
}
```

#### Three generations of encapsulation

You will see all three in real codebases:

| Approach | Example | Truly private? |
|---|---|---|
| Underscore convention | `this._balance` | ❌ No — convention only |
| Closure over local variable | factory function returning methods | ✅ Yes |
| `#` private fields (ES2022) | `this.#balance` | ✅ Yes, enforced by the language |

```javascript
// Closure-based privacy — the pre-ES2022 pattern, still valid
function createAccount(initial) {
  let balance = initial;               // not reachable from outside
  return {
    deposit: (n) => { balance += n; },
    getBalance: () => balance
  };
}
```

---

### 2. Inheritance

Inheritance lets a child class acquire the properties and methods of a parent using `extends`. The child constructor must call `super()` before touching `this`.

```javascript
class Animal {
  constructor(name) {
    this.name = name;
  }

  makeSound() {
    console.log(`${this.name} makes a sound.`);
  }
}

class Dog extends Animal {
  constructor(name, breed) {
    super(name);      // Executes the parent constructor
    this.breed = breed;
  }
}

const myDog = new Dog("Rex", "German Shepherd");
myDog.makeSound();    // Rex makes a sound.
```

#### `super` works in methods too, not just constructors

Use it to *extend* parent behavior instead of replacing it entirely:

```javascript
class Bird extends Animal {
  makeSound() {
    super.makeSound();          // run the parent version first
    console.log("...then chirps.");
  }
}
```

#### Extending built-ins — the practical case

Subclassing `Error` is extremely common in Node/Express apps:

```javascript
class AppError extends Error {
  constructor(message, statusCode = 500) {
    super(message);
    this.statusCode = statusCode;
    this.name = this.constructor.name;
    Error.captureStackTrace(this, this.constructor); // Node: clean stack trace
  }
}

class NotFoundError extends AppError {
  constructor(resource) {
    super(`${resource} not found`, 404);
  }
}

throw new NotFoundError("Campground");
```

#### Faking abstract classes

JavaScript has no `abstract` keyword. Two common workarounds:

```javascript
class Shape {
  constructor() {
    if (new.target === Shape) {
      throw new TypeError("Shape is abstract and cannot be instantiated directly");
    }
  }

  area() {
    throw new Error(`${this.constructor.name} must implement area()`);
  }
}
```

`new.target` is the constructor that was actually called with `new` — `undefined` for a plain function call.

---

### 3. Polymorphism

Polymorphism lets child classes provide their own implementation of a method already defined on the parent (**method overriding**).

```javascript
class Cat extends Animal {
  makeSound() {
    console.log(`${this.name} meows!`);
  }
}

const animals = [new Animal("Generic Creature"), new Cat("Whiskers")];
animals.forEach(animal => animal.makeSound());
// Generic Creature makes a sound.
// Whiskers meows!
```

#### JavaScript has no method overloading

Unlike Java or C++, you cannot define two methods with the same name and different signatures — the second definition simply replaces the first. Emulate it with default parameters, rest args, or an options object:

```javascript
class Logger {
  log(message, { level = "info", timestamp = true } = {}) {
    const prefix = timestamp ? `[${new Date().toISOString()}] ` : "";
    console.log(`${prefix}${level.toUpperCase()}: ${message}`);
  }
}
```

#### Duck typing

Because JavaScript is dynamically typed, polymorphism doesn't require a shared base class at all. Anything with a `makeSound()` method works in the loop above. This is often more idiomatic than building an inheritance hierarchy just to satisfy a type check.

#### Overriding built-in behavior

Objects participate in language operations through well-known methods:

```javascript
class Money {
  constructor(amount) { this.amount = amount; }

  toString()          { return `$${this.amount.toFixed(2)}`; }
  valueOf()           { return this.amount; }
  toJSON()            { return { amount: this.amount }; }
  [Symbol.iterator]() { /* makes the object usable with for...of and spread */ }
}

`${new Money(5)}`;          // "$5.00"
new Money(5) + new Money(3); // 8
```

---

### 4. Abstraction

Abstraction hides complex implementation details and exposes only the essential operations. In JavaScript this means designing clean public interfaces and concealing internal logic behind private fields or module boundaries.

```javascript
class CoffeeMachine {
  #waterTemp = 0; // Hidden internal detail

  #heatWater() {
    this.#waterTemp = 90;
    return true;
  }

  // Simple public interface over a multi-step process
  brew() {
    if (this.#heatWater()) {
      console.log("Your premium coffee is ready!");
    }
  }
}

new CoffeeMachine().brew(); // Your premium coffee is ready!
```

---

## ⚡ The `this` Problem

The single biggest source of OOP bugs in JavaScript. `this` is determined by **how a function is called**, not where it was defined.

```javascript
class Button {
  constructor(label) {
    this.label = label;
  }

  handleClick() {
    console.log(this.label);
  }
}

const btn = new Button("Save");
const handler = btn.handleClick;
handler();  // ❌ TypeError — `this` is undefined (class bodies are strict mode)
```

Three fixes:

```javascript
// 1. Bind in the constructor
constructor(label) {
  this.label = label;
  this.handleClick = this.handleClick.bind(this);
}

// 2. Arrow function as a class field — auto-bound, but per-instance (costs memory)
handleClick = () => {
  console.log(this.label);
};

// 3. Wrap at the call site
element.addEventListener("click", () => btn.handleClick());
```

Rule of thumb: regular methods for shared behavior, arrow class fields only when the method will be detached and passed around as a callback.

---

## 🧬 Prototypal Underpinnings

Behind the scenes, JavaScript has no real static classes. Every object holds an internal link to another object — its **prototype**. If a property isn't found on the object itself, the engine searches upward through the prototype chain until it finds it or hits `null`.

```javascript
const vehicle = { wheels: 4 };
const truck = Object.create(vehicle); // vehicle becomes truck's prototype

console.log(truck.wheels); // 4 — inherited via the prototype chain
```

Mapping the sugar to the machinery:

| Class syntax | What actually happens |
|---|---|
| `class Car {}` | A constructor function with a `.prototype` object |
| `accelerate() {}` in the body | `Car.prototype.accelerate = function () {}` |
| `class Dog extends Animal` | `Object.setPrototypeOf(Dog.prototype, Animal.prototype)` |
| `super(name)` | `Animal.call(this, name)` |
| `static compare() {}` | `Car.compare = function () {}` (on the constructor, not the prototype) |
| `#privateField` | No prototype equivalent — genuinely new capability |

### Where `class` is genuinely more than sugar

- Class bodies always run in strict mode.
- Classes are in the temporal dead zone — no usable hoisting.
- Calling a class without `new` throws a `TypeError`.
- `#` private fields have no pre-ES6 equivalent.

---

## 🔀 Composition Over Inheritance

Deep inheritance chains are the classic OOP failure mode: a change in a base class ripples unpredictably, and real-world entities rarely fit a single tree. Prefer building objects from smaller capabilities.

### Mixins

```javascript
const CanFly = (Base) => class extends Base {
  fly() { console.log(`${this.name} is flying`); }
};

const CanSwim = (Base) => class extends Base {
  swim() { console.log(`${this.name} is swimming`); }
};

class Duck extends CanFly(CanSwim(Animal)) {}

const d = new Duck("Donald");
d.fly();  d.swim();  d.makeSound();
```

### Plain composition

Often simpler and easier to test:

```javascript
class Order {
  constructor(items, paymentProcessor, notifier) {
    this.items = items;
    this.payment = paymentProcessor;   // injected dependency
    this.notifier = notifier;
  }

  async checkout() {
    await this.payment.charge(this.total());
    await this.notifier.send("Order confirmed");
  }
}
```

This is dependency injection — the same idea as swapping a Tableau data source or a repository layer. It keeps `Order` testable with fake collaborators.

---

## 🧩 Alternatives to Classes

Classes are not mandatory. Two lighter patterns you'll see constantly in Node codebases:

```javascript
// Factory function — no `new`, no `this`, closure-based privacy
function createUser(name) {
  let loginCount = 0;
  return {
    name,
    login() { loginCount++; },
    get logins() { return loginCount; }
  };
}

// Module pattern — a single exported object of related functions
export const userService = {
  async findById(id) { /* ... */ },
  async create(data) { /* ... */ }
};
```

Reach for a class when you need many instances each holding their own state. Reach for a module object when you need one collection of related functions.

---

## ⚖️ When to Use OOP in JavaScript

| Paradigm | Best used for | Key strengths |
|---|---|---|
| **Object-Oriented (OOP)** | Complex entity relationships, UI components, repeated stateful things (shopping carts, game characters, DB models) | High modularity, intuitive mapping to real-world structure |
| **Functional (FP)** | Heavy data transformation, pure math, utility libraries, async pipelines | Immutability, predictable output, no side effects |

In practice, idiomatic modern JavaScript mixes both: classes for entities and services, functional style (`map`/`filter`/`reduce`, pure helpers) for the data flowing through them.

---

## Common Pitfalls Checklist

- [ ] **Losing `this`** when passing a method as a callback → bind or wrap.
- [ ] **Arrow functions as prototype methods** → they have no own `this`.
- [ ] **Shared mutable state on the prototype** → arrays/objects on `.prototype` are shared by every instance. Put state in the constructor.
- [ ] **Forgetting `super()`** before using `this` in a subclass constructor → `ReferenceError`.
- [ ] **Inheritance depth beyond 2–3 levels** → switch to composition.
- [ ] **Mutating shared objects** → `Object.freeze()` for genuine constants (shallow only).
- [ ] **Overusing classes for stateless utilities** → a plain exported function is clearer.
- [ ] **Assuming `instanceof` is reliable across realms** (iframes, separate Node contexts) → it isn't; use duck typing or `Symbol.hasInstance`.

---

## Quick Reference

| Concept | Syntax |
|---|---|
| Define a class | `class Foo { }` |
| Constructor | `constructor(args) { }` |
| Instance method (on prototype) | `bar() { }` |
| Instance field (per instance) | `bar = value;` |
| Private field / method | `#bar` / `#bar() { }` |
| Getter / setter | `get bar() { }` / `set bar(v) { }` |
| Static member | `static bar` / `static bar() { }` |
| Static init block | `static { }` |
| Inheritance | `class Child extends Parent { }` |
| Call parent constructor | `super(args)` |
| Call parent method | `super.method()` |
| Detect direct instantiation | `new.target` |
| Freeze an object | `Object.freeze(obj)` |